In [ ]:
import geopandas as gpd
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from rasterstats import zonal_stats
import rasterio

FARMS_PATH = "GeoFiles/CSB/CSB_AZ_cleaned.shp"
RASTER_DIR = "GeoFiles/ET/Reprojected/"
OUT_DIR = "GeoFiles/ET/FarmWise/"

YEARS = range(2019, 2025)  # 2019–2024


gdf = gpd.read_file(FARMS_PATH)
print(f"Loaded {len(gdf)} farms")

In [ ]:
for year in YEARS:
    raster_path = f"{RASTER_DIR}Reprojected_ET_{year}.tif"
    out_path = f"{OUT_DIR}Farm_ETmm_{year}.shp"

    print(f"\nProcessing year {year} ...")

    # Read raster 
    with rasterio.open(raster_path) as src:
        nodata = src.nodata
        arr = src.read(1).astype(float)
        transform = src.transform

        # Precompute valid pixels (for fallback fill)
        valid_mask = arr != nodata
        if not valid_mask.any():
            raise ValueError(f"No valid pixels found in {raster_path}")
        fill_value = float(np.nanmedian(arr[valid_mask]))
    # Run zonal stats (mean ET depth in mm) for each farm
    stats = zonal_stats(
        gdf,
        arr,
        affine=transform,
        stats=["mean"],
        nodata=nodata,
        all_touched=True
    )
    df_stats = pd.DataFrame(stats)
    df_stats.index = gdf.index

    # Attach results
    gdf[f"ETmm_{year}"] = df_stats["mean"]

    # Rescue NaNs using nearest valid pixel value
    nan_idx = gdf[gdf[f"ETmm_{year}"].isna()].index
    if len(nan_idx) > 0:
        gdf.loc[nan_idx, f"ETmm_{year}"] = fill_value
    nan_after = gdf[f"ETmm_{year}"].isna().sum()
    print(f"NaN farms after rescue: {nan_after}")

    # Keep only key fields for saving
    out_gdf = gdf[['CSBID', 'CSBACRES', 'CNTY', 'geometry', f'ETmm_{year}']].copy()

    # Save shapefile
    out_gdf.to_file(out_path, driver="ESRI Shapefile")
    print(f"Saved {out_path}")

# Quick visualization for the last year
plt.figure(figsize=(8,5))
gdf[f"ETmm_{YEARS[-1]}"].hist(bins=50, edgecolor="black")
plt.title(f"Distribution of Farm Mean ET Depth (mm) - {YEARS[-1]}")
plt.xlabel("ET Depth (mm)")
plt.ylabel("Number of Farms")
plt.show()
